# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO

before = len(df)
df = df.drop_duplicates()
removed = before - len(df)

log("TODO 1", "Dropped duplicate rows", removed)


[TODO 1] Dropped duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
# TODO

df['price'] = (
    df['price']
    .str.replace('$', '', regex=False)
    .astype(float)
)

log("TODO 2", "Removed $ and converted price to float", len(df))


[TODO 2] Removed $ and converted price to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO

df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

before = len(df)

df = df.dropna(subset=['qty'])
df = df[df['qty'] >= 0]

removed = before - len(df)

log("TODO 3", "Converted qty to numeric and dropped missing/negative quantities", removed)

[TODO 3] Converted qty to numeric and dropped missing/negative quantities (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse


print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

before = df['item'].nunique()

df['item'] = df['item'].map(ITEM_MAP)

after = df['item'].nunique()

log(
    "TODO 4",
    "Canonicalized item names",
    before - after
)

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[TODO 4] Canonicalized item names (3 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
# TODO

### TODO 5 — normalize category

print(df['category'].value_counts())

CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear'
}

before = df['category'].nunique()

df['category'] = df['category'].map(CATEGORY_MAP)

after = df['category'].nunique()

log(
    "TODO 5",
    "Normalized category names and classified Apparel as Merch",
    before - after
)

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[TODO 5] Normalized category names and classified Apparel as Merch (3 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [9]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].nunique() == 3
assert df['category'].nunique() == 3
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [10]:
# TODO


df['revenue'] = df['qty'] * df['price']

revenue_by_category = (
    df.groupby('category')['revenue']
      .sum()
      .sort_values(ascending=False)
      .round(2)
)

print("Revenue by category:")
print(revenue_by_category)

overall_total = df['revenue'].sum()

print(f"\nOverall total revenue: ${overall_total:.2f}")

Revenue by category:
category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64

Overall total revenue: $4740.00


**What I would tell the vendor:** _..._ I would tell the vendor to stock more food because it generated the most revenue in the cleaned dataset.

### TODO 8 — read back your log

In [11]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,TODO 1,Dropped duplicate rows,15
1,TODO 2,Removed $ and converted price to float,300
2,TODO 3,Converted qty to numeric and dropped missing/n...,25
3,TODO 4,Canonicalized item names,3
4,TODO 5,Normalized category names and classified Appar...,3


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

_your answer here_